# Age-kernel heatmap (PIC tasks)

Computes kernel deviation (Panel A) + realized effect (Panel B) for each PIC task.
All figure styling is in this notebook — edit and re-run the plot cells.

In [7]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Markdown
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable


def _ensure_repo_on_path() -> Path:
    candidates = []
    try:
        candidates.append(Path(__vsc_ipynb_file__).resolve().parent)
    except NameError:
        pass
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    for base in candidates:
        for p in [base, *base.parents]:
            if (p / "model_new" / "figures" / "fig_age_kernel_heatmap.py").is_file():
                if str(p) not in sys.path:
                    sys.path.insert(0, str(p))
                return p
    raise ModuleNotFoundError("repo root with model_new/ not found")


REPO = _ensure_repo_on_path()
from model_new.figures.fig_age_kernel_heatmap import (
    DEFAULT_OUT,
    REPO_ROOT,
    compute_figure_data,
    days_from_tau,
)
from model_new import diagnostics as D

print("REPO =", REPO)

REPO = /home/suraj/Git/Age-conditioned-pediatric-EHR


## Config

In [8]:
RUN_DIR = REPO_ROOT / "model_new" / "run" / "kernel_s0_072420260946"
CKPT = "epoch_011.pt"
PIC_ROOT = REPO_ROOT / "data" / "tensorized" / "pic"
OUT_DIR = DEFAULT_OUT
EVAL_SPLIT = "pic_test"   # pic_test | pic_val | pic_train

PIC_TASKS = ["pneumonia", "mortality", "los_gt7", "heart_malformations"]

AGE_MAX = 18.0
BAND_TABLE = "pediatric"
MAX_BATCHES = 20
BATCH_SIZE = 12
PIC_SAMPLE_WINDOWS = 800
SEED = 0

SITE = "encoder"   # "encoder" | "pooling" | "both"

## Style knobs (edit these)

In [9]:
FIG_W, FIG_H = 7.0, 2.4
LEFT, RIGHT, BOTTOM, TOP = 0.08, 0.98, 0.20, 0.88
W_GAP = 0.04
WIDTH_A, WIDTH_B = 0.58, 0.42   # relative; renormalized below

CMAP_A, CMAP_B = "RdBu_r", "Reds"
N_CONTOUR = 5
COLORBAR_A = "additive attention-logit bias (nats)"
COLORBAR_B = "mean max|Δlogit| (nats)"

AGE_TICKS = (0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 15.0, 18.0)
LAG_TICKS = (
    ("1h", 1 / 24),
    ("1d", 1.0),
    ("1w", 7.0),
    ("1mo", 30.0),
    ("1y", 365.25),
    ("5y", 5 * 365.25),
)

SHOW_PIC_OVERLAY = True
PIC_COLOR, PIC_LS = "#196f3d", ":"

TITLE_A = "A  kernel deviation"
TITLE_B = "B  realized max|Δlogit|"
XLABEL_LAG = "inter-event lag"
YLABEL_AGE = "age (years)"

SAVE_DPI = 300

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.titlesize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def age_to_u(a):
    return np.log1p(a)

## Plot function (A | B only)

In [10]:
def _age_strip(ax, hist, *, color, age_max):
    edges = np.asarray(hist.get("edges") or [], dtype=np.float64)
    fracs = np.asarray(hist.get("fractions") or [], dtype=np.float64)
    if edges.size < 2 or fracs.size != edges.size - 1:
        return
    mx = float(fracs.max()) or 1.0
    y0, y1 = ax.get_ylim()
    for i, f in enumerate(fracs):
        lo, hi = float(edges[i]), float(min(edges[i + 1], age_max))
        if hi <= lo or lo >= age_max:
            continue
        ya0 = (age_to_u(lo) - y0) / (y1 - y0)
        ya1 = (age_to_u(hi) - y0) / (y1 - y0)
        ax.add_patch(Rectangle(
            (0.945, ya0), 0.045 * f / mx, max(ya1 - ya0, 1e-4),
            transform=ax.transAxes, color=color, alpha=0.4, linewidth=0,
            clip_on=False, zorder=6,
        ))


def plot_ab(data, site_name: str, *, title_suffix: str = "",
            save_stem: str | None = None, show: bool = True):
    s = data["sites"][site_name]
    ages, lag_days = data["ages"], data["lag_days"]
    delta, panel_b = s["delta"], s["panel_b"]
    age_max = float(data["age_max"])
    ticks = tuple(a for a in AGE_TICKS if a <= age_max + 1e-9) or (0.0, age_max)

    u = age_to_u(ages)
    abs_d = np.abs(delta)
    lim = float(np.quantile(abs_d, 0.99)) if abs_d.size else 1e-12
    lim = max(lim, 1e-12)

    fig = plt.figure(figsize=(FIG_W, FIG_H))
    w = np.array([WIDTH_A, WIDTH_B], dtype=float)
    w = w / w.sum() * (RIGHT - LEFT - W_GAP)
    ax_a = fig.add_axes([LEFT, BOTTOM, w[0], TOP - BOTTOM])
    ax_b = fig.add_axes([LEFT + w[0] + W_GAP, BOTTOM, w[1], TOP - BOTTOM])

    # Panel A
    X, Y = np.meshgrid(lag_days, u)
    pcm = ax_a.pcolormesh(
        X, Y, delta, cmap=CMAP_A, shading="auto",
        norm=Normalize(vmin=-lim, vmax=lim),
    )
    levels = np.linspace(-lim, lim, N_CONTOUR + 2)[1:-1]
    if np.any(np.abs(levels) > 0):
        ax_a.contour(X, Y, delta, levels=levels, colors="black",
                     linewidths=0.35, alpha=0.7)

    ax_a.set_xscale("log")
    ax_a.set_xticks([d for _, d in LAG_TICKS])
    ax_a.set_xticklabels([lab for lab, _ in LAG_TICKS])
    ax_a.set_xlabel(XLABEL_LAG)
    ax_a.set_yticks([age_to_u(a) for a in ticks])
    ax_a.set_yticklabels([str(int(a)) if float(a).is_integer() else str(a) for a in ticks])
    ax_a.set_ylabel(YLABEL_AGE)
    ax_a.set_xlim(lag_days.min(), lag_days.max())
    ax_a.set_ylim(u.min(), u.max())
    ax_a.set_title(f"{TITLE_A}{title_suffix}", loc="left", pad=2)

    if SHOW_PIC_OVERLAY:
        for ov in data["overlays"]:
            if ov["name"] != "PIC":
                continue
            for qk, lab in (("0.05", "5th"), ("0.5", "50th"), ("0.95", "95th")):
                d_q = max(float(days_from_tau(float(ov["tau_percentiles"][qk]))),
                          float(lag_days.min()))
                ax_a.axvline(d_q, color=PIC_COLOR, linestyle=PIC_LS,
                             linewidth=0.7, alpha=0.85, zorder=5)
            d50 = max(float(days_from_tau(float(ov["tau_percentiles"]["0.5"]))),
                      float(lag_days.min()))
            ax_a.text(d50, u.max(), "PIC", color=PIC_COLOR, fontsize=5.5,
                      ha="center", va="bottom", clip_on=False)
            _age_strip(ax_a, ov.get("age_histogram") or {},
                       color=PIC_COLOR, age_max=age_max)

    cax = make_axes_locatable(ax_a).append_axes("top", size="6%", pad=0.08)
    cb = fig.colorbar(pcm, cax=cax, orientation="horizontal")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cb.set_label(COLORBAR_A, fontsize=7)
    cb.ax.tick_params(labelsize=6)

    # Panel B
    means = np.asarray(panel_b["mean_max_abs_delta_logit"], dtype=np.float64)
    counts = np.asarray(panel_b["n"], dtype=np.int64)
    finite = means[np.isfinite(means)]
    vmax_b = max(float(np.nanmax(finite)) if finite.size else 1e-12, 1e-12)
    im = ax_b.imshow(means, aspect="auto", origin="lower", cmap=CMAP_B,
                     vmin=0.0, vmax=vmax_b, interpolation="nearest")
    ax_b.set_xticks(range(len(panel_b["quartile_labels"])))
    ax_b.set_xticklabels(panel_b["quartile_labels"])
    ax_b.set_yticks(range(len(panel_b["band_names"])))
    ax_b.set_yticklabels(panel_b["band_names"])
    ax_b.set_xlabel("τ-spread quartile")
    ax_b.set_title(TITLE_B, loc="left", pad=2)
    for i in range(means.shape[0]):
        for j in range(means.shape[1]):
            n, val = int(counts[i, j]), means[i, j]
            txt = f"{val:.2f}\nn={n}" if n and math.isfinite(val) else f"n={n}"
            ax_b.text(j, i, txt, ha="center", va="center", fontsize=5.0)
    cax_b = make_axes_locatable(ax_b).append_axes("top", size="6%", pad=0.08)
    cb_b = fig.colorbar(im, cax=cax_b, orientation="horizontal")
    cax_b.xaxis.set_ticks_position("top")
    cb_b.set_label(COLORBAR_B, fontsize=7)
    cb_b.ax.tick_params(labelsize=6)

    if save_stem is not None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        for ext in ("pdf", "png"):
            path = OUT_DIR / f"{save_stem}.{ext}"
            kw = {"format": ext, "bbox_inches": "tight"}
            if ext == "png":
                kw["dpi"] = SAVE_DPI
            fig.savefig(path, **kw)
            print("wrote", path)

    if show:
        display(fig)
    plt.close(fig)

## Compute + plot each PIC task

One figure per task. Filenames: `fig_age_kernel_heatmap_<task>.png` (encoder) and `..._<task>_pooling.png`.

In [11]:
results = {}

for task in PIC_TASKS:
    display(Markdown(f"### `{task}`"))
    data = compute_figure_data(
        RUN_DIR, CKPT,
        eval_split=EVAL_SPLIT,
        pic_root=PIC_ROOT,
        pic_task=task,
        out_dir=OUT_DIR,
        batch_size=BATCH_SIZE,
        max_batches=MAX_BATCHES,
        pic_sample_windows=PIC_SAMPLE_WINDOWS,
        seed=SEED,
        age_max=AGE_MAX,
        band_table=BAND_TABLE,
    )
    results[task] = data
    enc, pool = data["encoder_site"], data["pooling_site"]
    meta = data["meta"]
    display(Markdown(
        f"arm=`{meta['arm']}` · `{Path(meta['checkpoint_path']).name}` · "
        f"eval=`{data['eval_label']}` · "
        f"max|Δ| enc=`{data['sites'][enc]['numbers']['max_abs_delta']:.3g}` / "
        f"pool=`{data['sites'][pool]['numbers']['max_abs_delta']:.3g}`"
    ))

    suffix = f" ({task})"
    if SITE in ("encoder", "both"):
        plot_ab(data, enc, title_suffix=suffix,
                save_stem=f"fig_age_kernel_heatmap_{task}", show=True)
    if SITE in ("pooling", "both"):
        plot_ab(data, pool, title_suffix=suffix,
                save_stem=f"fig_age_kernel_heatmap_{task}_pooling", show=True)

# Keep pneumonia as the default stem aliases (optional convenience copies)
if "pneumonia" in results and SITE in ("encoder", "both"):
    plot_ab(results["pneumonia"], results["pneumonia"]["encoder_site"],
            title_suffix=" (pneumonia)",
            save_stem="fig_age_kernel_heatmap", show=False)

### `pneumonia`

arm=`kernel` · `epoch_011.pt` · eval=`pic/pneumonia/test` · max|Δ| enc=`42.3` / pool=`16.2`

wrote /home/suraj/Git/Age-conditioned-pediatric-EHR/model_new/figures/out/fig_age_kernel_heatmap_pneumonia.pdf
wrote /home/suraj/Git/Age-conditioned-pediatric-EHR/model_new/figures/out/fig_age_kernel_heatmap_pneumonia.png


<Figure size 700x240 with 4 Axes>

### `mortality`

...

### `heart_malformations`

arm=`kernel` · `epoch_011.pt` · eval=`pic/heart_malformations/test` · max|Δ| enc=`42.3` / pool=`16.2`

wrote /home/suraj/Git/Age-conditioned-pediatric-EHR/model_new/figures/out/fig_age_kernel_heatmap_heart_malformations.pdf
wrote /home/suraj/Git/Age-conditioned-pediatric-EHR/model_new/figures/out/fig_age_kernel_heatmap_heart_malformations.png


<Figure size 700x240 with 4 Axes>

wrote /home/suraj/Git/Age-conditioned-pediatric-EHR/model_new/figures/out/fig_age_kernel_heatmap.pdf
wrote /home/suraj/Git/Age-conditioned-pediatric-EHR/model_new/figures/out/fig_age_kernel_heatmap.png


## Save numbers JSON (all tasks)

In [ ]:
payload = {
    "checkpoint": results[PIC_TASKS[0]]["meta"]["checkpoint_path"],
    "arm": results[PIC_TASKS[0]]["meta"]["arm"],
    "epoch": results[PIC_TASKS[0]]["meta"]["epoch"],
    "age_max": AGE_MAX,
    "band_table": BAND_TABLE,
    "tasks": {},
}
for task, data in results.items():
    enc, pool = data["encoder_site"], data["pooling_site"]
    payload["tasks"][task] = {
        "eval_split": data["eval_label"],
        "encoder": data["sites"][enc]["numbers"],
        "pooling": data["sites"][pool]["numbers"],
    }

out_json = OUT_DIR / "age_kernel_heatmap_numbers.json"
D.write_json(out_json, payload)
print("wrote", out_json)
print("tasks:", list(payload["tasks"]))